In [12]:
import torch
import torch.nn as nn
from einops import rearrange

# Patch Embedding
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels, patch_size=16, emb_dim=256):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, emb_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = rearrange(x, 'b e h w -> b (h w) e')
        return x


# ViT Encoder
class ViTEncoder(nn.Module):
    def __init__(self, dim=256, depth=6, heads=8):
        super().__init__()
        layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=depth)

    def forward(self, x):
        return self.encoder(x)


# Conv Block
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU()
        )

    def forward(self, x):
        return self.block(x)


# Decoder
class Decoder(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.up1 = ConvBlock(emb_dim, 128)
        self.up2 = ConvBlock(128, 64)
        self.up3 = ConvBlock(64, 32)
        self.final = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        x = nn.functional.interpolate(x, scale_factor=2)
        x = self.up1(x)

        x = nn.functional.interpolate(x, scale_factor=2)
        x = self.up2(x)

        x = nn.functional.interpolate(x, scale_factor=2)
        x = self.up3(x)

        return torch.sigmoid(self.final(x))


# Full Model
class ViT_UNetPP(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.patch = PatchEmbedding(in_channels)
        self.vit = ViTEncoder()
        self.decoder = Decoder(256)

    def forward(self, x):
        patches = self.patch(x)
        encoded = self.vit(patches)

        B, N, E = encoded.shape
        size = int(N ** 0.5)

        x = encoded.permute(0, 2, 1).reshape(B, E, size, size)
        out = self.decoder(x)

        return out

In [13]:
import os
import torch
import netCDF4 as nc
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

from model import ViT_UNetPP

ModuleNotFoundError: No module named 'model'